# 鶯歌工商 Mini AI Assistant & YKVS demo Colab｜優化版（穩定性／結構／效能）

保留原本 NCHU workshop 的 **Express → MCP Client → Python FastMCP → `search_school_info`** 架構與所有既有資料（79 筆場館、240+ 筆人員、57 班、4 份生輔規定、教務處規定索引、21 週行事曆、`get_weather` MCP），資料內容維持原有版本；本次同步優化前端介面與問答分流。

這一版針對三件事做了整體重構：

- **穩定性／錯誤處理**：所有安裝、啟動、tunnel 建立都改用共用的 `run()` / `wait_for_http()` 小工具，失敗會自動重試並印出清楚的錯誤訊息；資料驗證改成「一次列出所有不符合項目」而不是卡在第一個 assert；伺服器與 tunnel 啟動前都會先確認是否已經在正常運作，避免不必要的重啟。
- **程式碼品質／結構**：新增一個「共用設定與工具」格，把路徑常數與重複邏輯（執行指令、等待服務啟動、雜湊比對）集中管理，後面每一格都變短、也更容易單獨看懂在做什麼。
- **執行效率**：npm / uv 安裝、資料包安裝、localtunnel／qrcode 安裝，以及既有 tunnel，只要內容沒有變動就會直接跳過，重新執行整本 Notebook 時明顯更快。

**資料不必逐項上傳。**只需把這一個 `.ipynb` 上傳/開啟於 Colab，照順序執行各格；資料包已內嵌。第一次仍需 Colab 下載 workshop 原始程式並安裝依賴。請先在 Colab Secrets 建立 `GEMINI_API_KEY`，僅供一般生成式問答使用；星期與日期為本機計算，明確校務資料走 MCP 快查，開放性問題不會被強制導向法規。

`english_center.json` 已含 21 週（2026-08-30 至 2027-01-23）。Notebook 會檢查週數、79 筆場館、240 筆人員紀錄、57 班與 4 份生輔規定，不再要求另外上傳行事曆 JSON。


## 1｜共用設定與工具
用途：集中定義專案路徑常數，以及後面每一格都會用到的小工具函式——執行指令並自動重試的 `run()`、等待服務啟動的 `wait_for_http()`、用來判斷「內容沒變就跳過」的雜湊比對。之後每一格都會變得更短、更好懂。


In [ ]:
import base64
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path

# ── 專案路徑（全 Notebook 共用一份，只需維護這裡）──────────────────────────
REPO = Path('/content/nchu-mcp-workshop-2026')
PROJECT = REPO / 'mini-project'
DATA = PROJECT / 'mcp-server-py/data'
PACK = DATA / 'ykvs_mvp_source_pack/data'
EXPRESS_URL = 'http://127.0.0.1:3000'

MARKERS = Path('/content/.ykvs_markers')
MARKERS.mkdir(parents=True, exist_ok=True)

# ── 共用小工具：讓下面每一格更短、錯誤訊息更一致，並支援「重跑時跳過已完成步驟」──
def banner(msg: str, ok: bool = True) -> None:
    print(f"{'✅' if ok else '⚠️'} {msg}")

def run(cmd, cwd=None, env=None, check=True, retries=1, retry_wait=2):
    """執行 shell 指令；失敗時自動重試，最終仍失敗才印出完整輸出並丟出例外。"""
    result = None
    for attempt in range(1, retries + 1):
        result = subprocess.run(
            cmd, cwd=cwd, env=env, shell=isinstance(cmd, str),
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            return result
        if attempt < retries:
            time.sleep(retry_wait)
    if check and result is not None and result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f'指令執行失敗（已重試 {retries} 次）：{cmd}')
    return result

def http_ok(url: str, timeout: float = 2.0) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return 200 <= r.status < 400
    except Exception:
        return False

def wait_for_http(url: str, timeout: int = 40, interval: float = 1.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if http_ok(url):
            return True
        time.sleep(interval)
    return False

def file_hash(*paths: Path) -> str:
    h = hashlib.sha256()
    for p in paths:
        if p.exists():
            h.update(p.read_bytes())
    return h.hexdigest()

def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def marker_matches(name: str, value: str) -> bool:
    f = MARKERS / name
    return f.exists() and f.read_text() == value

def write_marker(name: str, value: str) -> None:
    (MARKERS / name).write_text(value)

banner('共用設定與工具已就緒（路徑、run/wait_for_http、標記快取機制）。')


## 2｜取得 workshop 並安裝依賴
用途：從原 workshop 專案建立既有 Node/Express 與 Python MCP 環境；保留原專案，不另建查詢伺服器。已安裝過的部分（repo、node、uv、npm/uv 依賴）會自動偵測並跳過，重新執行 Notebook 時更快；下載或安裝失敗時會自動重試。


In [ ]:
if not PROJECT.exists():
    banner('複製 workshop 專案（首次執行）…')
    run(['git', 'clone', '-q', 'https://github.com/UDICatNCHU/nchu-mcp-workshop-2026', str(REPO)], retries=2)
else:
    banner('workshop 專案已存在，略過複製。')

if shutil.which('node') is None:
    banner('安裝 Node.js 20…')
    run(
        'curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1 '
        '&& sudo apt-get install -y nodejs >/dev/null 2>&1',
        retries=2,
    )
else:
    banner('Node.js 已安裝，略過。')

if shutil.which('uv') is None:
    banner('安裝 uv…')
    run('curl -LsSf https://astral.sh/uv/install.sh | sh', retries=2)
    os.environ['PATH'] = str(Path.home() / '.local/bin') + ':' + os.environ['PATH']
else:
    banner('uv 已安裝，略過。')

# 只有在 package.json / pyproject.toml 內容改變時才重新安裝依賴，
# 讓「重新執行全部」在依賴沒變的情況下快非常多。
node_manifest = PROJECT / 'backend-node/package.json'
py_manifest = PROJECT / 'mcp-server-py/pyproject.toml'
deps_signature = file_hash(node_manifest, py_manifest)

if marker_matches('deps_installed', deps_signature):
    banner('Node / Python 依賴未變動，略過重新安裝。')
else:
    banner('安裝 Node / Python 依賴…')
    run(['npm', 'install', '--silent'], cwd=PROJECT / 'backend-node', retries=2)
    run(['uv', 'sync', '--quiet'], cwd=PROJECT / 'mcp-server-py', retries=2)
    write_marker('deps_installed', deps_signature)

banner(f'workshop 專案及依賴準備完成：{PROJECT}')


## 3｜安裝你的 YKVS 一體化更新資料

用途：直接套用你的版本並整合學姊新增功能；新版 `ykvs.html` 內建漸層常用標籤、柔和點擊漣漪與重新設計的輸入框，移除左側「快速分類」欄並改成單欄乾淨版，**不需要 `beautify.html`**；問答也會區分「校務資料快查」與「開放性 AI 問答」。資料包內容以雜湊比對，若和上次安裝的完全相同就直接跳過，重跑更快；寫入前仍會做路徑檢查（防止跳脫專案目錄）並保留原檔備份。


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)

PACK_B64 = """..."""

pack_signature = text_hash(PACK_B64)

if marker_matches('data_pack_installed', pack_signature):
    banner('YKVS 一體化資料內容未變動，略過重新安裝（沿用先前結果）。')
else:
    banner('安裝 YKVS 一體化更新資料…')
    archive = zipfile.ZipFile(io.BytesIO(base64.b64decode(PACK_B64)))
    allowed = {
        'backend-node/server.js', 'backend-node/llm-client-openai.js', 'backend-node/quick-reply.js',
        'mcp-server-py/hello_tool.py', 'mcp-server-py/data/english_center.json', 'web/ykvs.html',
    }
    allowed_prefixes = ('mcp-server-py/data/ykvs_mvp_source_pack/', 'scripts/')
    entries = [n for n in archive.namelist() if n in allowed or n.startswith(allowed_prefixes)]
    if not entries:
        raise RuntimeError('內嵌資料包沒有可安裝檔案')

    # 覆蓋前先備份既有檔案，避免資料遺失
    backup = Path('/content/ykvs_backup') / datetime.now().strftime('%Y%m%d_%H%M%S')
    for rel in entries:
        target = PROJECT / rel
        if target.is_file():
            dest = backup / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, dest)

    # 寫入前先檢查路徑，拒絕任何跳脫專案目錄的項目（防止 zip 路徑穿越攻擊）
    for rel in entries:
        target = (PROJECT / rel).resolve()
        if PROJECT.resolve() not in target.parents:
            raise ValueError(f'拒絕寫入專案目錄之外：{rel}')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(rel))

    write_marker('data_pack_installed', pack_signature)
    banner(f'YKVS 一體化資料安裝完成：{len(entries)} 個檔案')
    banner(f'備份位置：{backup}')

banner('保留原 workshop config.json、web/index.html 與其他 MCP tools。')


## 4｜檢查資料與 MCP 設定
用途：驗證 79 筆場館、至少 240 筆人員紀錄、57 班、4 份法規、21 週行事曆，以及原 workshop 的 `hello_tool`、`weather_tool` 註冊。這一版會把所有檢查項目**一次列出**（每項標示 ✅／❌），而不是卡在第一個不符合的地方，方便一次看清楚哪些需要重新安裝資料包。


In [ ]:
staff = json.loads((PACK / 'staff_extensions.json').read_text(encoding='utf-8'))
classes = json.loads((PACK / 'class_extensions.json').read_text(encoding='utf-8'))
calendar = json.loads((DATA / 'english_center.json').read_text(encoding='utf-8'))
laws = sorted((PACK / 'regulations').glob('*.md'))
academic_laws = json.loads((PACK / 'additional_regulations.json').read_text(encoding='utf-8')).get('records', [])
config = json.loads((PROJECT / 'config.json').read_text(encoding='utf-8'))
mcp_servers = config.get('mcpServers', {})

counts = {
    'staff': len(staff.get('分機紀錄', [])),
    'venues': len(staff.get('場館分機', [])),
    'classes': len(classes.get('班級分機', {})),
    'broadcast': len(classes.get('年級廣播', [])),
    'regulations': len(laws),
    'academic_regulations': len(academic_laws),
    'calendar_weeks': len(calendar.get('calendar_weeks', [])),
}

# 把所有檢查收集起來一次跑完，而不是遇到第一個錯誤就中斷、
# 這樣重新安裝資料包前就能一次看到「所有」不符合的項目。
checks = [
    ('人員資料 ≥ 240 筆', counts['staff'] >= 240, counts['staff']),
    ('場館分機 = 79 筆', counts['venues'] == 79, counts['venues']),
    ('班級分機 = 57 班', counts['classes'] == 57, counts['classes']),
    ('生輔組規定 = 4 份', counts['regulations'] == 4, counts['regulations']),
    ('教務處規定索引 ≥ 8 項', counts['academic_regulations'] >= 8, counts['academic_regulations']),
    ('115-1 行事曆 = 21 週', counts['calendar_weeks'] == 21, counts['calendar_weeks']),
    ('web/ykvs.html 存在', (PROJECT / 'web/ykvs.html').exists(), None),
    ('scripts/pdf_to_regulations_json.py 存在', (PROJECT / 'scripts/pdf_to_regulations_json.py').exists(), None),
    ('hello_tool 已於 config.json 註冊', 'hello_tool' in mcp_servers, None),
    ('weather_tool 已於 config.json 註冊', 'weather_tool' in mcp_servers, None),
]

print('資料核對：', counts)
failed = []
for label, ok, value in checks:
    detail = f'（目前：{value}）' if value is not None else ''
    banner(f'{label}{detail}', ok=ok)
    if not ok:
        failed.append(label)

if failed:
    raise AssertionError(
        '以下檢查未通過，請重新執行「安裝 YKVS 一體化更新資料」那一格：\n- ' + '\n- '.join(failed)
    )

print(f'教務處規定索引：{len(academic_laws)} 項')
banner('檢查通過：新版介面、教務處快查、MCP 工具設定、法規索引及行事曆都已備妥。')


## 5｜讀取 Gemini Secret
用途：讓一般自由問答使用原 Gemini + MCP 流程。分機／法規／行事曆走 MCP 快查；星期日期本機計算；天氣沿用 `get_weather` MCP。金鑰只從 Colab Secrets 讀取，不印出、不寫入檔案。


In [ ]:
from google.colab import userdata

key = userdata.get('GEMINI_API_KEY')
if not key:
    raise ValueError('請先在 Colab 左側「秘密金鑰」建立 GEMINI_API_KEY，並允許此 Notebook 存取。')

os.environ['LLM_PROVIDER'] = 'openai'
os.environ['OPENAI_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
os.environ['OPENAI_API_KEY'] = key
os.environ['OPENAI_MODEL'] = 'gemini-3.1-flash-lite'
del key
banner('模型金鑰已讀取，內容不會顯示。')


## 6｜啟動原有 Express 與 MCP 服務
用途：重新啟動 workshop 既有 `server.js`，確認服務就緒；此步驟會連接設定檔中的 MCP servers。若伺服器已經在執行且回應正常（例如只是重跑這一格），會直接略過重啟，不會多花時間。


In [ ]:
if http_ok(EXPRESS_URL):
    # 伺服器已在執行且健康（例如重新執行 Notebook 但 Colab 連線未重啟），
    # 不需要重新啟動，省下重啟與再次等待的時間。
    banner('Express server 已在執行且回應正常，略過重新啟動。')
else:
    run(['bash', '-lc', "pkill -f '[n]ode server.js' || true"], check=False)
    log_path = Path('/content/server.log')
    log_file = log_path.open('w')
    server_proc = subprocess.Popen(
        ['node', 'server.js'],
        cwd=PROJECT / 'backend-node',
        env=os.environ.copy(),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    if not wait_for_http(EXPRESS_URL, timeout=40, interval=1):
        print(log_path.read_text(encoding='utf-8', errors='replace')[-5000:])
        raise RuntimeError('server 未在 40 秒內啟動；請查看上方錯誤記錄')
    banner('Express 啟動成功。')

log_path = Path('/content/server.log')
log_text = log_path.read_text(encoding='utf-8', errors='replace') if log_path.exists() else ''
mcp_lines = [l for l in log_text.splitlines() if 'search_school_info' in l or 'get_weather' in l or '[MCP]' in l]
print('MCP 工具記錄：')
print('\n'.join(mcp_lines) or '請確認 server log 中 hello_tool 與 weather_tool 已連線')


## 7｜開啟新版查詢頁（Colab 內建網址）
新版包含分組常用查詢標籤、柔和漸層、點擊漣漪，以及重新整理的輸入區。這個連結只能在目前的 Colab 執行階段內使用；要給手機掃描的外網網址請看下一格。


In [ ]:
from google.colab.output import eval_js
from IPython.display import display, HTML

base_url = eval_js('google.colab.kernel.proxyPort(3000)')
url_ykvs = base_url.rstrip('/') + '/ykvs.html'

display(HTML(f'''
    <div style="font-family: sans-serif; padding: 12px; border: 1px solid #e0e0e0; border-radius: 8px; background-color: #f8f9fa;">
        <p style="margin: 0 0 8px 0; font-weight: bold; color: #1a4b84;">✨ 鶯歌工商 Mini AI Assistant 新版介面已就緒！</p>
        <a href="{url_ykvs}" target="_blank" rel="noopener" style="padding: 8px 16px; background-color: #1a4b84; color: white; text-decoration: none; border-radius: 4px; display: inline-block;">
            點此開啟新版校務 AI 助理 (ykvs.html)
        </a>
    </div>
'''))


## 8｜建立外網 tunnel 與手機 QR Code

> 若手機顯示 **Bad Gateway**，通常代表舊的 localtunnel 已失效或 Colab 的 Express server 已停止。請重新執行第 6 格，再重新執行本格產生新的 QR Code；不要繼續使用舊 QR Code。

QR Code 會先等待並驗證 localtunnel 外網可連線後才產生；QR Code 編碼使用乾淨的 `/ykvs.html` 網址。快速標籤點擊時不再自動聚焦輸入框，因此不會把整個頁面跳到最下面；一般開放性問題不會因為包含成績、學分、學習歷程等詞就被強制導向法規。

這一版會先確認既有 tunnel 是否還活著、外網是否還連得到，能沿用就直接沿用，不用每次重跑都重新建立一次；localtunnel／qrcode 套件也只在尚未安裝時才會安裝。


In [ ]:
from IPython.display import display, HTML

# 只在尚未安裝時才安裝，避免每次重跑都重新裝一次
if shutil.which('lt') is None:
    banner('安裝 localtunnel…')
    run(['npm', 'install', '-g', 'localtunnel', '--silent'], retries=2)
try:
    import qrcode
except ImportError:
    banner('安裝 qrcode…')
    run(['pip', 'install', 'qrcode[pil]', '--quiet'], retries=2)
    import qrcode

if not http_ok(EXPRESS_URL):
    raise RuntimeError('本機 Express server 已停止，請先重新執行「啟動 Express 與 MCP 服務」那一格，再執行本格。')

def check_remote_ok(url: str, attempts: int = 12, interval: float = 1.0):
    """等待外網 tunnel 真正能取得網站再產生 QR Code，
    避免「Colab 顯示網址可以點，但手機一掃 QR Code 卻 Bad Gateway」的競態問題。"""
    status, error = None, ''
    for _ in range(attempts):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as r:
                status = r.status
                if 200 <= r.status < 400:
                    return True, status, error
        except Exception as exc:
            error = str(exc)
        time.sleep(interval)
    return False, status, error

def start_tunnel(port: int = 3000, attempts: int = 3, wait_s: int = 20):
    """啟動 localtunnel，取得對外網址；失敗會重試 attempts 次。"""
    last_output = ''
    for _ in range(attempts):
        proc = subprocess.Popen(
            ['npx', 'localtunnel', '--port', str(port)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            bufsize=1, start_new_session=True,
        )
        lines, url, deadline = [], '', time.time() + wait_s
        while time.time() < deadline:
            line = proc.stdout.readline()
            if line:
                lines.append(line.strip())
                m = re.search(r'https?://[^\s]+', line)
                if m and 'loca.lt' in m.group(0):
                    url = m.group(0).rstrip('/')
                    break
            if proc.poll() is not None:
                break
        last_output = '\n'.join(lines[-12:])
        if url and proc.poll() is None:
            return proc, url, ''
        try:
            proc.terminate()
        except Exception:
            pass
        time.sleep(1.5)
    return None, '', last_output

def tunnel_alive(proc) -> bool:
    return proc is not None and proc.poll() is None

# 沿用仍存活、且外網仍可連通的既有 tunnel，避免每次重跑都重新建立一次
old_proc = globals().get('tunnel_proc')
old_page = globals().get('tunnel_page', '')
reused = False
if tunnel_alive(old_proc) and old_page and http_ok(old_page):
    tunnel_proc, tunnel_page = old_proc, old_page
    banner(f'沿用仍然有效的既有 tunnel：{tunnel_page}')
    reused = True
else:
    if old_proc is not None:
        try:
            old_proc.terminate()
            old_proc.wait(timeout=3)
        except Exception:
            try:
                old_proc.kill()
            except Exception:
                pass

if not reused:
    banner('建立新的 localtunnel…')
    tunnel_proc, tunnel_url, last_output = start_tunnel(port=3000)
    if not tunnel_url:
        print('localtunnel 啟動紀錄：')
        print(last_output or '沒有取得 localtunnel URL。')
        raise RuntimeError('localtunnel 沒有取得可用網址。請重新執行本格；若仍失敗，先重新執行啟動 server 的格。')
    tunnel_page = tunnel_url + '/ykvs.html'

    remote_ok, remote_status, remote_error = check_remote_ok(tunnel_page)
    if not remote_ok:
        raise RuntimeError(
            'localtunnel 已取得網址，但外網尚未能正常連到 ykvs.html。'
            f'\n最後狀態：{remote_status or remote_error}'
            '\n請重新執行本格建立新的 tunnel。'
        )

try:
    ip_password = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=5).read().decode().strip()
except Exception:
    ip_password = '若手機出現密碼頁，請使用 Colab 顯示的 IP；若沒有密碼頁可直接略過。'

# QR Code 直接編碼乾淨網址，不加入查詢參數，
# 手機掃描器與一般瀏覽器都會進入完全相同的頁面。
qr = qrcode.QRCode(
    version=None,
    error_correction=qrcode.constants.ERROR_CORRECT_H,
    box_size=8,
    border=4,
)
qr.add_data(tunnel_page)
qr.make(fit=True)
img = qr.make_image(fill_color='#315d72', back_color='white')
buf = io.BytesIO()
img.save(buf, format='PNG')
qr_b64 = base64.b64encode(buf.getvalue()).decode()

html_card = f'''\
<div style="font-family:system-ui,-apple-system,'Noto Sans TC',sans-serif;padding:18px;border:1px solid #d8e4e6;border-radius:18px;background:linear-gradient(135deg,#fff,#f3f7f6);max-width:540px;text-align:center;box-shadow:0 8px 24px rgba(39,61,70,.06)">
<div style="font-size:18px;font-weight:750;color:#315d72;margin-bottom:6px">🎓 鶯歌工商校務 AI 助理</div>
<div style="font-size:13px;color:#71818a;margin-bottom:14px">掃描 QR Code，直接開啟最新版校務 AI 查詢頁</div>
<img src="data:image/png;base64,{qr_b64}" alt="QR Code" style="width:220px;height:220px;border:1px solid #dce5e7;padding:10px;border-radius:14px;background:#fff">
<div style="margin-top:12px;font-size:12px;color:#71818a">外網網址</div>
<a href="{tunnel_page}" target="_blank" style="display:inline-block;margin-top:4px;color:#315d72;word-break:break-all;font-size:13px">{tunnel_page}</a>
<div style="margin-top:12px;padding:9px 11px;border-radius:10px;background:#edf4f1;color:#5f747b;font-size:12px;line-height:1.6">若手機出現 Endpoint IP / Password 頁面，再輸入：<b style="color:#8a6f3d">{ip_password}</b><br>若沒有密碼頁，直接進入即可。</div>
</div>'''
display(HTML(html_card))
banner('localtunnel 已備妥（新建立或沿用既有），請使用這次卡片上的 QR Code。')
print('網址：', tunnel_page)


### 建議驗收題
1. 鶯歌工商生輔組分機有哪些？（841、611）
2. 請查教師姓名的單位、職稱和分機。
3. 教學大樓 6D 電腦教室分機是多少？
4. 學生在家因病不能到校，返校後幾天內要補辦請假？請附學生請假規則來源。
5. 115-1 行事曆 2026/10/6 有哪些行程？
6. 今天星期幾？（應由台灣時區本機快速計算）

若 Gemini 回覆遇到 503 或逾時，先用分機、法規、行事曆快查題驗收 MCP 路徑；一般生成式問答仍會受模型服務狀態與免費額度影響。

7. 學習歷程怎麼準備？（應進入一般 AI 問答，不應直接回傳法規）
8. 成績不好怎麼改善？（應進入一般 AI 問答，不應只查法規）
9. 我要找教務主任。（應進入校務資料快查，查人員／處室資訊）

8. 點擊「老師／分機／請假」等快速標籤：只帶入提問範例到輸入框，不應自動送出；可修改後再查詢。
9. 「段考沒參加怎麼辦？」、「學習歷程怎麼準備？」等開放問題：應先由 AI 自然語言理解，不應直接只回傳法規。
